# Pneumonia Detection from Chest X-Rays using Deep Learning

**Project Overview:**  
This project builds a binary classifier to detect pneumonia in chest X-ray images using:
1. Custom CNN architecture with batch normalization and dropout
2. VGG16 transfer learning with pretrained ImageNet weights

**Key Features:**
- Class imbalance handling via weighted loss and data augmentation
- Comprehensive medical metrics (Precision, Recall, ROC-AUC)
- Grad-CAM visualization for model interpretability
- Production-ready evaluation pipeline

**Dataset:** Chest X-Ray Images (Pneumonia) from Kaggle  
**Author:** Kommal Tariq  
**Date:** 2024-2025

## 1. Setup and Imports

In [ ]:
# Standard library imports
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Data handling
import numpy as np
import pandas as pd

# Deep learning framework
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TensorBoard
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Import custom modules
sys.path.append('..')
from models.cnn_models import (
    create_custom_cnn, create_vgg16_transfer_learning, 
    compile_model, get_model_summary
)
from utils.data_utils import (
    create_data_generators, calculate_class_weights,
    analyze_dataset_distribution, create_sample_visualization
)
from utils.visualization import (
    plot_training_history, evaluate_model_comprehensive,
    plot_confusion_matrix, plot_roc_curve, plot_precision_recall_curve,
    visualize_predictions, generate_gradcam_heatmaps
)

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Create directories
os.makedirs('../results', exist_ok=True)
os.makedirs('../models', exist_ok=True)

print("All imports successful!")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

# Configure GPU memory growth (prevents TensorFlow from allocating all GPU memory)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU memory growth enabled for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(e)

## 2. Configuration

In [ ]:
# Configuration dictionary
config = {
    # Data paths (adjust these to your dataset location)
    'train_dir': '../data/chest_xray/train',
    'val_dir': '../data/chest_xray/val',
    'test_dir': '../data/chest_xray/test',
    
    # Image preprocessing
    'img_size': (224, 224),      # Standard size for VGG16 and most CNNs
    'batch_size': 32,             # Number of images per training batch
    
    # Training parameters
    'epochs': 25,                 # Maximum number of epochs
    'learning_rate': 0.001,       # Initial learning rate
    'use_class_weights': True,    # Handle class imbalance
    'use_augmentation': True,     # Apply data augmentation
    
    # Model selection
    'model_type': 'both',         # 'custom', 'vgg16', or 'both'
    
    # Reproducibility
    'seed': 42
}

# Set random seeds
np.random.seed(config['seed'])
tf.random.set_seed(config['seed'])

print("Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

## 3. Dataset Analysis

**IMPORTANT NOTE FOR USERS:**  
Before running the cells below, you need to download the dataset:

1. Go to Kaggle: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
2. Click "Download" (requires free Kaggle account)
3. Extract the ZIP file to `../data/chest_xray/`
4. The folder structure should be:
   ```
   data/chest_xray/
   ├── train/
   │   ├── NORMAL/
   │   └── PNEUMONIA/
   ├── val/
   │   ├── NORMAL/
   │   └── PNEUMONIA/
   └── test/
       ├── NORMAL/
       └── PNEUMONIA/
   ```

In [ ]:
# Verify dataset exists
if not os.path.exists(config['train_dir']):
    print("❌ ERROR: Dataset not found!")
    print("Please download from: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia")
    print(f"Expected location: {config['train_dir']}")
else:
    print("✓ Dataset found!")
    
    # Analyze dataset distribution
    distribution_df = analyze_dataset_distribution(
        config['train_dir'],
        config['val_dir'],
        config['test_dir']
    )

## 4. Create Data Generators

In [ ]:
# Create data generators with augmentation
train_generator, val_generator, test_generator = create_data_generators(
    train_dir=config['train_dir'],
    val_dir=config['val_dir'],
    test_dir=config['test_dir'],
    img_size=config['img_size'],
    batch_size=config['batch_size'],
    augment=config['use_augmentation']
)

## 5. Visualize Sample Images

In [ ]:
# Show sample images from training set
create_sample_visualization(
    train_generator,
    num_samples=16,
    save_path='../results/training_samples.png'
)

## 6. Calculate Class Weights

In [ ]:
# Calculate class weights to handle imbalance
# This tells the model to pay more attention to the minority class
class_weights = None
if config['use_class_weights']:
    class_weights = calculate_class_weights(train_generator)

## 7. Build and Compile Models

In [ ]:
# Dictionary to store models and histories
models_dict = {}
histories_dict = {}

# Create Custom CNN
if config['model_type'] in ['custom', 'both']:
    print("\n" + "="*70)
    print("CREATING CUSTOM CNN MODEL")
    print("="*70)
    
    custom_model = create_custom_cnn(
        input_shape=(*config['img_size'], 3),
        num_classes=2
    )
    custom_model = compile_model(custom_model, learning_rate=config['learning_rate'])
    get_model_summary(custom_model, save_path='../results/custom_cnn_summary.txt')
    
    models_dict['Custom CNN'] = custom_model

# Create VGG16 Transfer Learning Model
if config['model_type'] in ['vgg16', 'both']:
    print("\n" + "="*70)
    print("CREATING VGG16 TRANSFER LEARNING MODEL")
    print("="*70)
    
    vgg16_model = create_vgg16_transfer_learning(
        input_shape=(*config['img_size'], 3),
        num_classes=2,
        freeze_base=True  # Freeze pretrained layers initially
    )
    vgg16_model = compile_model(vgg16_model, learning_rate=config['learning_rate'])
    get_model_summary(vgg16_model, save_path='../results/vgg16_summary.txt')
    
    models_dict['VGG16'] = vgg16_model

print(f"\n✓ Created {len(models_dict)} model(s): {list(models_dict.keys())}")

## 8. Define Training Callbacks

In [ ]:
def get_callbacks(model_name):
    """
    Create training callbacks for model optimization.
    
    Callbacks are functions that run during training:
    - ModelCheckpoint: Saves best model based on validation accuracy
    - EarlyStopping: Stops training if validation loss doesn't improve
    - ReduceLROnPlateau: Reduces learning rate when stuck
    
    Args:
        model_name: Name for saving checkpoints
    
    Returns:
        List of callback objects
    """
    callbacks = [
        # Save best model
        ModelCheckpoint(
            filepath=f'../models/{model_name}_best.h5',
            monitor='val_accuracy',      # Metric to monitor
            mode='max',                   # Save when metric increases
            save_best_only=True,          # Only save if better than previous
            verbose=1
        ),
        
        # Stop training if no improvement
        EarlyStopping(
            monitor='val_loss',
            patience=7,                   # Wait 7 epochs before stopping
            restore_best_weights=True,    # Restore weights from best epoch
            verbose=1
        ),
        
        # Reduce learning rate when stuck
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,                   # Multiply LR by 0.5
            patience=3,                   # Wait 3 epochs before reducing
            min_lr=1e-7,                  # Don't go below this LR
            verbose=1
        )
    ]
    
    return callbacks

## 9. Train Models

In [ ]:
# Train each model
for model_name, model in models_dict.items():
    print("\n" + "="*70)
    print(f"TRAINING {model_name}")
    print("="*70 + "\n")
    
    # Get callbacks
    callbacks = get_callbacks(model_name.replace(' ', '_'))
    
    # Train model
    history = model.fit(
        train_generator,
        epochs=config['epochs'],
        validation_data=val_generator,
        class_weight=class_weights,    # Apply class weights
        callbacks=callbacks,
        verbose=1
    )
    
    # Store history
    histories_dict[model_name] = history
    
    print(f"\n✓ {model_name} training completed!")
    print(f"Best validation accuracy: {max(history.history['val_accuracy'])*100:.2f}%")

print("\n" + "="*70)
print("ALL MODELS TRAINED SUCCESSFULLY!")
print("="*70)

## 10. Plot Training History

In [ ]:
# Plot training curves for each model
for model_name, history in histories_dict.items():
    plot_training_history(
        history,
        save_path=f'../results/{model_name.replace(" ", "_")}_training_history.png'
    )

## 11. Comprehensive Evaluation on Test Set

In [ ]:
# Evaluate each model on test set
test_results = {}

for model_name, model in models_dict.items():
    print("\n" + "="*70)
    print(f"EVALUATING {model_name} ON TEST SET")
    print("="*70)
    
    # Comprehensive evaluation
    results = evaluate_model_comprehensive(
        model,
        test_generator,
        class_names=['NORMAL', 'PNEUMONIA']
    )
    
    test_results[model_name] = results
    
    print("\n" + "-"*70)

## 12. Create Results Comparison Table

In [ ]:
# Create comprehensive comparison table
comparison_data = []

for model_name in models_dict.keys():
    comparison_data.append({
        'Model': model_name,
        'Test Accuracy (%)': f"{test_results[model_name]['accuracy']*100:.2f}",
        'Precision (%)': f"{test_results[model_name]['precision']*100:.2f}",
        'Recall (%)': f"{test_results[model_name]['recall']*100:.2f}",
        'F1-Score (%)': f"{test_results[model_name]['f1_score']*100:.2f}",
        'Best Val Acc (%)': f"{max(histories_dict[model_name].history['val_accuracy'])*100:.2f}"
    })

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*80)
print("MODEL COMPARISON - TEST SET RESULTS")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Save to CSV
comparison_df.to_csv('../results/model_comparison.csv', index=False)
print("\nResults saved to ../results/model_comparison.csv")

## 13. Confusion Matrices

In [ ]:
# Plot confusion matrix for each model
for model_name, results in test_results.items():
    plot_confusion_matrix(
        results['y_true'],
        results['y_pred'],
        class_names=['NORMAL', 'PNEUMONIA'],
        save_path=f'../results/{model_name.replace(" ", "_")}_confusion_matrix.png'
    )

## 14. ROC Curves

In [ ]:
# Plot ROC curve for each model
auc_scores = {}

for model_name, results in test_results.items():
    print(f"\nGenerating ROC curve for {model_name}...")
    auc_score = plot_roc_curve(
        results['y_true'],
        results['y_pred_probs'],
        save_path=f'../results/{model_name.replace(" ", "_")}_roc_curve.png'
    )
    auc_scores[model_name] = auc_score

## 15. Precision-Recall Curves

In [ ]:
# Plot Precision-Recall curve for each model
# Important for medical imaging where we care about both precision and recall
ap_scores = {}

for model_name, results in test_results.items():
    print(f"\nGenerating Precision-Recall curve for {model_name}...")
    ap_score = plot_precision_recall_curve(
        results['y_true'],
        results['y_pred_probs'],
        save_path=f'../results/{model_name.replace(" ", "_")}_pr_curve.png'
    )
    ap_scores[model_name] = ap_score

## 16. Visualize Sample Predictions

In [ ]:
# Visualize predictions for best model
best_model_name = max(test_results.items(), key=lambda x: x[1]['accuracy'])[0]
best_model = models_dict[best_model_name]

print(f"Visualizing predictions for best model: {best_model_name}\n")

visualize_predictions(
    best_model,
    test_generator,
    num_samples=16,
    save_path=f'../results/{best_model_name.replace(" ", "_")}_predictions.png'
)

## 17. Grad-CAM Visualization (Model Interpretability)

Grad-CAM shows which parts of the X-ray the model focuses on when making predictions.  
This is **critical for medical AI** to verify the model is looking at clinically relevant features.

In [ ]:
# Generate Grad-CAM heatmaps for best model
print(f"Generating Grad-CAM visualizations for {best_model_name}...\n")

generate_gradcam_heatmaps(
    best_model,
    test_generator,
    num_samples=8,
    save_path=f'../results/{best_model_name.replace(" ", "_")}_gradcam.png'
)

## 18. Performance Metrics Summary

In [ ]:
# Create comprehensive metrics summary
print("\n" + "="*80)
print("COMPREHENSIVE METRICS SUMMARY")
print("="*80 + "\n")

summary_data = []
for model_name in models_dict.keys():
    summary_data.append({
        'Model': model_name,
        'Accuracy': f"{test_results[model_name]['accuracy']*100:.2f}%",
        'Precision': f"{test_results[model_name]['precision']*100:.2f}%",
        'Recall': f"{test_results[model_name]['recall']*100:.2f}%",
        'F1-Score': f"{test_results[model_name]['f1_score']*100:.2f}%",
        'AUC-ROC': f"{auc_scores[model_name]:.4f}",
        'Avg Precision': f"{ap_scores[model_name]:.4f}"
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print("\n" + "="*80)

# Save to CSV
summary_df.to_csv('../results/comprehensive_metrics.csv', index=False)
print("\nComprehensive metrics saved to ../results/comprehensive_metrics.csv")

## 19. Clinical Interpretation

Understanding metrics in medical context:
- **Recall (Sensitivity)**: % of actual pneumonia cases correctly identified  
  *High recall is critical - we don't want to miss pneumonia cases*

- **Precision**: % of positive predictions that are actually pneumonia  
  *High precision reduces false alarms*

- **Specificity**: % of normal cases correctly identified as normal  
  *Important to avoid unnecessary treatment*

In [ ]:
from sklearn.metrics import confusion_matrix

print("\n" + "="*80)
print("CLINICAL INTERPRETATION (Best Model)")
print("="*80 + "\n")

# Calculate specificity for best model
y_true = test_results[best_model_name]['y_true']
y_pred = test_results[best_model_name]['y_pred']

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

sensitivity = tp / (tp + fn)  # Same as recall
specificity = tn / (tn + fp)
ppv = tp / (tp + fp)  # Positive Predictive Value (same as precision)
npv = tn / (tn + fn)  # Negative Predictive Value

print(f"Model: {best_model_name}\n")
print(f"Sensitivity (Recall):     {sensitivity*100:.2f}%")
print(f"  → Out of 100 pneumonia cases, we correctly identify {sensitivity*100:.0f}")
print(f"  → We miss {(1-sensitivity)*100:.0f} cases (False Negatives)\n")

print(f"Specificity:              {specificity*100:.2f}%")
print(f"  → Out of 100 normal cases, we correctly identify {specificity*100:.0f}")
print(f"  → We incorrectly flag {(1-specificity)*100:.0f} as pneumonia (False Positives)\n")

print(f"Positive Predictive Value: {ppv*100:.2f}%")
print(f"  → When model says 'pneumonia', it's correct {ppv*100:.0f}% of the time\n")

print(f"Negative Predictive Value: {npv*100:.2f}%")
print(f"  → When model says 'normal', it's correct {npv*100:.0f}% of the time\n")

print("="*80)
print("\n💡 Clinical Recommendation:")
if sensitivity >= 0.95:
    print("✓ High sensitivity - Good for screening (few missed cases)")
else:
    print("⚠ Consider using as screening tool with radiologist confirmation")

if specificity >= 0.90:
    print("✓ High specificity - Low false positive rate")
else:
    print("⚠ May generate false alarms - requires clinical judgment")

print("="*80)

## 20. Final Summary and Recommendations

In [ ]:
print("\n" + "="*80)
print("PROJECT SUMMARY")
print("="*80 + "\n")

print("📊 Dataset: Chest X-Ray Pneumonia")
print(f"   - Training samples: {train_generator.samples:,}")
print(f"   - Validation samples: {val_generator.samples:,}")
print(f"   - Test samples: {test_generator.samples:,}")

print(f"\n🏆 Best Model: {best_model_name}")
print(f"   - Test Accuracy: {test_results[best_model_name]['accuracy']*100:.2f}%")
print(f"   - AUC-ROC: {auc_scores[best_model_name]:.4f}")
print(f"   - Sensitivity: {sensitivity*100:.2f}%")
print(f"   - Specificity: {specificity*100:.2f}%")

print("\n🔧 Techniques Used:")
print("   ✓ Data augmentation (rotations, flips, shifts, zoom)")
print("   ✓ Class weight balancing for imbalanced dataset")
print("   ✓ Batch normalization and dropout regularization")
print("   ✓ Transfer learning with VGG16 (ImageNet weights)")
print("   ✓ Learning rate scheduling and early stopping")
print("   ✓ Comprehensive medical metrics (Sensitivity/Specificity)")
print("   ✓ Grad-CAM for model interpretability")

print("\n📁 Generated Artifacts:")
artifacts = [
    'Training history plots',
    'Confusion matrices',
    'ROC curves with AUC scores',
    'Precision-Recall curves',
    'Grad-CAM heatmaps (model attention)',
    'Sample predictions with confidence scores',
    'Model comparison tables',
    'Saved model checkpoints (.h5 files)'
]
for artifact in artifacts:
    print(f"   - {artifact}")

print("\n💡 Future Improvements:")
improvements = [
    "Try EfficientNet or ResNet architectures",
    "Implement ensemble methods (combine predictions)",
    "Add test-time augmentation for better accuracy",
    "Fine-tune VGG16 top layers after initial training",
    "Experiment with different class weight strategies",
    "Add uncertainty estimation (model confidence intervals)",
    "Deploy as Flask/FastAPI REST API for inference"
]
for i, improvement in enumerate(improvements, 1):
    print(f"   {i}. {improvement}")

print("\n" + "="*80)
print("✅ Notebook execution completed successfully!")
print("="*80)